# Ensemble/Voting Classification in Python with Scikit-Learn
ref：https://www.kaggle.com/c/titanic/submit

In [191]:
import pandas as pd
import numpy as np
import warnings

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.model_selection import train_test_split, KFold, cross_val_score

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier, ExtraTreesClassifier

In [192]:
training_data = pd.read_csv("data/train.csv")
testing_data = pd.read_csv("data/test.csv")
def get_nulls(training, testing):
    print("Training Data:")
    print(pd.isnull(training).sum())
    print("\nTesting Data:")
    print(pd.isnull(testing).sum())

get_nulls(training_data, testing_data)

#rint("training_data.shape=",training_data.shape)
#print("training_data=",training_data.head())


Training Data:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Testing Data:
PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64


In [193]:
# Drop the cabin column, as there are too many missing values
# Drop the ticket numbers too, as there are too many categories
# Drop names as they won't really help predict survivors

training_data.drop(labels=['Cabin','Ticket','Name'],axis=1,inplace=True)
testing_data.drop(labels=['Cabin','Ticket','Name'],axis=1,inplace=True)



print("training_data",training_data.head())
print("\ntesting_data",testing_data.head())


# Taking the mean/average value would be impacted by the skew
# so we should use the median value to impute missing values
training_data["Age"].fillna(training_data["Age"].median(),inplace=True)
testing_data["Age"].fillna(testing_data["Age"].median(),inplace=True)

training_data["Embarked"].fillna("S",inplace=True)
testing_data["Fare"].fillna(testing_data["Fare"].median(),inplace=True)


get_nulls(training_data, testing_data)

training_data    PassengerId  Survived  Pclass     Sex   Age  SibSp  Parch     Fare Embarked
0            1         0       3    male  22.0      1      0   7.2500        S
1            2         1       1  female  38.0      1      0  71.2833        C
2            3         1       3  female  26.0      0      0   7.9250        S
3            4         1       1  female  35.0      1      0  53.1000        S
4            5         0       3    male  35.0      0      0   8.0500        S

testing_data    PassengerId  Pclass     Sex   Age  SibSp  Parch     Fare Embarked
0          892       3    male  34.5      0      0   7.8292        Q
1          893       3  female  47.0      1      0   7.0000        S
2          894       2    male  62.0      0      0   9.6875        Q
3          895       3    male  27.0      0      0   8.6625        S
4          896       3  female  22.0      1      1  12.2875        S
Training Data:
PassengerId    0
Survived       0
Pclass         0
Sex            0
A

/var/folders/_x/j4r1s_6d1y5_tx91ctfmvdbm0000gn/T/ipykernel_17686/1196712619.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  training_data["Age"].fillna(training_data["Age"].median(),inplace=True)
/var/folders/_x/j4r1s_6d1y5_tx91ctfmvdbm0000gn/T/ipykernel_17686/1196712619.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which w

In [194]:
# Fit the encoder on the data (Feature: Sex)
#print("training_data[Sex]=",training_data["Sex"])
#print("*************************************************")
#print("training_data=\n",training_data)

encoder_1 = LabelEncoder()
encoder_1.fit(training_data["Sex"])

print("encoder_1 class=\n")
print(encoder_1.classes_)

# Transform and replace training data
#(A.1)
training_sex_encoded = encoder_1.transform(training_data["Sex"])
training_data["Sex"] = training_sex_encoded

#print("#################################################")
#print("training_data=\n",training_data)


#print("*************************************************")
#print("Test data=\n",testing_data)
#(A.2)
test_sex_encoded = encoder_1.transform(testing_data["Sex"])
testing_data["Sex"] = test_sex_encoded
#print("*************************************************")
#print("Test data=\n",testing_data)


#____________________________________________________________________________
# Fit the encoder on the data (Feature: Embarked)
encoder_2 = LabelEncoder()
encoder_2.fit(training_data["Embarked"])
#print("encoder_2 class=\n")
#print(encoder_2.classes_)
#print(training_data["Embarked"].head(10))


#(A.3)
training_embarked_encoded = encoder_2.transform(training_data["Embarked"])
training_data["Embarked"] = training_embarked_encoded
#print("#################################################")
#print("training_data=\n",training_data)

#(A.4)
testing_embarked_encoded = encoder_2.transform(testing_data["Embarked"])
testing_data["Embarked"] = testing_embarked_encoded
#print("TTTTTTTTTTTTTTTTTTTTTTTTTTTTTTT")
#print("training_data=\n",training_data)

# Any value we want to reshape needs be turned into array first
#print("training_dataAge1=\n",training_data["Age"])


ages_train = np.array(training_data["Age"]).reshape(-1, 1)
fares_train= np.array(training_data["Fare"]).reshape(-1, 1)


ages_test = np.array(testing_data["Age"]).reshape(-1, 1)
fares_test= np.array(testing_data["Fare"]).reshape(-1, 1)

print("ages_train=1\n",ages_train[0])


#For average=0, stddev=1
# Scaler takes arrays
scaler = StandardScaler()

training_data["Age"] = scaler.fit_transform(ages_train)
training_data["Fare"] = scaler.fit_transform(fares_train)

testing_data["Age"] = scaler.fit_transform(ages_test)
testing_data["Fare"] = scaler.fit_transform(fares_test)
#print("ages_train=2\n",ages_train[0])


encoder_1 class=

['female' 'male']
ages_train=1
 [22.]


In [ ]:
# Now to select our training/testing data
X_features = training_data.drop(labels=['PassengerId', 'Survived'], axis=1)
y_labels = training_data['Survived']

print(X_features.head(5))
print(y_labels.head(5))

# Make the train/test data from validation

X_train, X_val, y_train, y_val = train_test_split(X_features, y_labels, test_size=0.1,random_state=12)
#X_train, X_test, y_train, y_test = train_test_split(
''' 
X_train  # 訓練用特徵 90%
X_val    # 驗證用特徵 10%
y_train  # 訓練用答案 90%
y_val    # 驗證用答案 10%
'''


   Pclass  Sex       Age  SibSp  Parch      Fare  Embarked
0       3    1 -0.565736      1      0 -0.502445         2
1       1    0  0.663861      1      0  0.786845         0
2       3    0 -0.258337      0      0 -0.488854         2
3       1    0  0.433312      1      0  0.420730         2
4       3    1  0.433312      0      0 -0.486337         2
0    0
1    1
2    1
3    1
4    0
Name: Survived, dtype: int64


' \nX_train  # 訓練用特徵，90%\nX_val    # 驗證用特徵，10%\ny_train  # 訓練用答案，90%\ny_val    # 驗證用答案，10%\n'

## Simple Averaging Approach

In [196]:
LogReg_clf = LogisticRegression()
DTree_clf = DecisionTreeClassifier()
SVC_clf = SVC()

LogReg_clf.fit(X_train, y_train)
DTree_clf.fit(X_train, y_train)
SVC_clf.fit(X_train, y_train)

LogReg_pred = LogReg_clf.predict(X_val)
DTree_pred = DTree_clf.predict(X_val)
SVC_pred = SVC_clf.predict(X_val)

averaged_preds = (LogReg_pred + DTree_pred + SVC_pred)//3
acc = accuracy_score(y_val, averaged_preds)
print(acc)

0.8


## Bagging Classification Example

In [197]:

#logreg_bagging_model=BaggingClassifier(base_estimator=LogReg_clf, n_estimators=100,random_state=12)
#dtree_bagging_model=BaggingClassifier(base_estimator=DTree_clf, n_estimators=100,random_state=12)
logreg_bagging_model=BaggingClassifier(estimator=LogReg_clf, n_estimators=100,random_state=12)
dtree_bagging_model=BaggingClassifier(estimator=DTree_clf, n_estimators=100,random_state=12)

random_forest=RandomForestClassifier(n_estimators=100,random_state=12)
extra_trees=ExtraTreesClassifier(n_estimators=100,random_state=12)


def bagging_ensemble(model):
    k_folds = KFold(n_splits=20, random_state=12,shuffle=True)
    results = cross_val_score(model, X_train, y_train, cv=k_folds)
    print(results.mean())


bagging_ensemble(logreg_bagging_model)
bagging_ensemble(dtree_bagging_model)
bagging_ensemble(random_forest)
bagging_ensemble(extra_trees)



0.7964634146341464
0.8238719512195123
0.8101219512195122
0.7963719512195122


## Boosting Classification Example

In [198]:
k_folds = KFold(n_splits=20, random_state=12,shuffle=True)
num_estimators = [20, 40, 60, 80, 100]

for i in num_estimators:
    ada_boost = AdaBoostClassifier(n_estimators=i, random_state=12)
    results = cross_val_score(ada_boost,X_train,y_train,cv=k_folds)
    print("Results for {} estimators:".format(i))
    print(results.mean())
    

Results for 20 estimators:
0.8064634146341463
Results for 40 estimators:
0.8089024390243903
Results for 60 estimators:
0.8051829268292684
Results for 80 estimators:
0.8039329268292683
Results for 100 estimators:
0.8051829268292684


## voting\Stacking Classification Example

In [ ]:
voting_clf = VotingClassifier(estimators=[('SVC', SVC_clf), ('DTree', DTree_clf), ('LogReg', LogReg_clf)], voting='hard')
voting_clf.fit(X_train, y_train)
preds = voting_clf.predict(X_val)

'''
X_train  # 訓練用特徵 90%
X_val    # 驗證用特徵 10%
y_train  # 訓練用答案 90%
y_val    # 驗證用答案 10%
'''

#print(len(X_val)) 90
#print(len(preds)) 90


acc = accuracy_score(y_val, preds)
l_loss = log_loss(y_val, preds)
f1 = f1_score(y_val, preds)

print("Accuracy is: " + str(acc))
print("Log Loss is: " + str(l_loss))
print("F1 Score is: " + str(f1))

Accuracy is: 0.8333333333333334
Log Loss is: 6.0072755648528595
F1 Score is: 0.7761194029850746


In [200]:
print("X_train=\n",X_train[:10])



X_train=
      Pclass  Sex       Age  SibSp  Parch      Fare  Embarked
715       3    1 -0.796286      0      0 -0.494391         2
319       1    0  0.817561      1      1  2.059694         0
829       1    0  2.508257      0      0  0.962353         2
79        3    0  0.049062      0      0 -0.397241         2
484       1    1 -0.335187      1      0  1.185430         0
116       3    1  3.161481      0      0 -0.492378         1
177       1    0  1.586059      0      0 -0.070304         0
10        3    0 -1.949034      1      1 -0.312172         2
862       1    0  1.432359      0      0 -0.126345         2
374       3    0 -2.025883      3      1 -0.224083         2


In [201]:
print("testing_data=\n",testing_data[:10])

test=testing_data.drop(labels=['PassengerId'],axis=1)
test.shape


testing_data=
    PassengerId  Pclass  Sex       Age  SibSp  Parch      Fare  Embarked
0          892       3    1  0.386231      0      0 -0.497413         1
1          893       3    0  1.371370      1      0 -0.512278         2
2          894       2    1  2.553537      0      0 -0.464100         1
3          895       3    1 -0.204852      0      0 -0.482475         2
4          896       3    0 -0.598908      1      1 -0.417492         2
5          897       3    1 -1.229397      0      0 -0.472391         2
6          898       3    0  0.031581      0      0 -0.500999         1
7          899       2    1 -0.283663      1      1 -0.117894         2
8          900       3    0 -0.914152      0      0 -0.508169         0
9          901       3    1 -0.677719      2      0 -0.204838         2


(418, 7)

In [ ]:
#print("test=\n",test)

submission_df=pd.DataFrame(columns=['PassengerId','Survived'])
submission_df['PassengerId']=testing_data['PassengerId']
voting_clf.predict(test)

'''
preds = voting_clf.predict(X_val)


X_train  # 訓練用特徵 90%
X_val    # 驗證用特徵 10%
y_train  # 訓練用答案 90%
y_val    # 驗證用答案 10%
'''

print(len(preds))
#submission_df['Survived']=preds
submission_df['Survived']=preds

submission_df.to_csv('submissions.csv')

print(len(submission_df))



90


ValueError: Length of values (90) does not match length of index (418)